In [ ]:
"""# Problem Statement

You're working as an AI engineer at an e-commerce company. The business team wants to predict which customers are likely to churn (stop making purchases). Your job is to build a comprehensive training dataset using the company's transaction database.

The database has the following tables:

**customers**: Contains customer information

- customer_id (PRIMARY KEY)
- customer_name
- email
- registration_date
- country

**orders**: Contains order information

- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY)
- order_date
- order_total
- status (completed, cancelled, refunded)

**products**: Contains product catalog

- product_id (PRIMARY KEY)
- product_name
- category
- price

**order_items**: Contains items in each order

- order_item_id (PRIMARY KEY)
- order_id (FOREIGN KEY)
- product_id (FOREIGN KEY)
- quantity
- item_price

**customer_support_tickets**: Contains support interactions

- ticket_id (PRIMARY KEY)
- customer_id (FOREIGN KEY)
- ticket_date
- issue_type (delivery, refund, product_quality, other)
- resolution_time_hours
- satisfaction_rating (1-5, can be NULL)

Your task is to create a training dataset with the following requirements:

1. Calculate features for each customer as of a specific cutoff date (2024-06-01)
2. Include only customers who registered at least 90 days before the cutoff date
3. Create a label: customer churned if they haven't made a purchase in the 60 days following the cutoff date
4. Generate these features:
    - Total number of orders
    - Total amount spent
    - Average order value
    - Days since last order (as of cutoff date)
    - Number of unique product categories purchased
    - Percentage of cancelled orders
    - Number of support tickets
    - Average support ticket satisfaction rating
    - Days active (days between first and last order)
    - Recency, Frequency, Monetary (RFM) scores
5. Handle NULL values appropriately
6. Exclude customers with less than 2 orders (not enough history)"""

In [9]:
import psycopg2

conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="Roman@333",
    host="localhost",
    port=5432
)

conn.autocommit = True
cursor = conn.cursor()

cursor.execute("""CREATE TABLE customers (
    customer_id SERIAL PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    email VARCHAR(100) NOT NULL,
    registration_date DATE NOT NULL,
    country VARCHAR(50)
);
""")

In [10]:
cursor.execute("""CREATE TABLE orders (
    order_id SERIAL PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(customer_id),
    order_date DATE NOT NULL,
    order_total NUMERIC(10,2) NOT NULL,
    status VARCHAR(20) NOT NULL
);
""")

In [11]:
cursor.execute("""CREATE TABLE products (
    product_id SERIAL PRIMARY KEY,
    product_name VARCHAR(200) NOT NULL,
    category VARCHAR(50) NOT NULL,
    price NUMERIC(10,2) NOT NULL
);
""")

In [12]:
cursor.execute("""CREATE TABLE order_items (
    order_item_id SERIAL PRIMARY KEY,
    order_id INTEGER REFERENCES orders(order_id),
    product_id INTEGER REFERENCES products(product_id),
    quantity INTEGER NOT NULL,
    item_price NUMERIC(10,2) NOT NULL
);
""")

In [13]:
cursor.execute("""CREATE TABLE customer_support_tickets (
    ticket_id SERIAL PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(customer_id),
    ticket_date DATE NOT NULL,
    issue_type VARCHAR(50) NOT NULL,
    resolution_time_hours INTEGER,
    satisfaction_rating INTEGER
);
""")

In [14]:
cursor.execute("""INSERT INTO customers (customer_name, email, registration_date, country) VALUES
('Himanshu Sharma', 'himanshu@email.com', '2023-01-15', 'India'),
('Sarah Johnson', 'sarah@email.com', '2023-02-20', 'USA'),
('Carlos Martinez', 'carlos@email.com', '2023-03-10', 'Spain'),
('Emma Wilson', 'emma@email.com', '2024-05-01', 'UK'),
('Michael Chen', 'michael@email.com', '2023-04-05', 'Canada'),
('Priya Patel', 'priya@email.com', '2023-01-25', 'India'),
('James Brown', 'james@email.com', '2023-02-15', 'USA');
""")

cursor.execute("""
               INSERT INTO products (product_name, category, price) VALUES
('Laptop Pro', 'Electronics', 1299.99),
('Wireless Mouse', 'Electronics', 29.99),
('Office Chair', 'Furniture', 249.99),
('Desk Lamp', 'Furniture', 45.99),
('Python Book', 'Books', 39.99),
('Coffee Maker', 'Appliances', 89.99),
('Headphones', 'Electronics', 149.99),
('Standing Desk', 'Furniture', 499.99);
               """)
cursor.execute("""
               INSERT INTO orders (customer_id, order_date, order_total, status) VALUES
(1, '2023-02-01', 1329.98, 'completed'),
(1, '2023-04-15', 45.99, 'completed'),
(1, '2024-03-20', 249.99, 'completed'),
(1, '2024-05-10', 149.99, 'completed'),
(2, '2023-03-05', 89.99, 'completed'),
(2, '2023-06-20', 1299.99, 'completed'),
(2, '2024-01-15', 45.99, 'cancelled'),
(3, '2023-04-10', 249.99, 'completed'),
(3, '2023-08-22', 499.99, 'completed'),
(3, '2024-02-14', 39.99, 'completed'),
(5, '2023-05-01', 1329.98, 'completed'),
(5, '2023-07-15', 149.99, 'completed'),
(5, '2023-09-30', 89.99, 'completed'),
(5, '2024-06-15', 249.99, 'completed'),
(6, '2023-02-10', 499.99, 'completed'),
(6, '2023-05-20', 149.99, 'completed'),
(6, '2024-04-05', 89.99, 'completed'),
(7, '2023-03-01', 1299.99, 'completed'),
(7, '2023-12-10', 45.99, 'completed');
               """)

cursor.execute("""
               INSERT INTO order_items (order_id, product_id, quantity, item_price) VALUES
(1, 1, 1, 1299.99),
(1, 2, 1, 29.99),
(2, 4, 1, 45.99),
(3, 3, 1, 249.99),
(4, 7, 1, 149.99),
(5, 6, 1, 89.99),
(6, 1, 1, 1299.99),
(7, 4, 1, 45.99),
(8, 3, 1, 249.99),
(9, 8, 1, 499.99),
(10, 5, 1, 39.99),
(11, 1, 1, 1299.99),
(11, 2, 1, 29.99),
(12, 7, 1, 149.99),
(13, 6, 1, 89.99),
(14, 3, 1, 249.99),
(15, 8, 1, 499.99),
(16, 7, 1, 149.99),
(17, 6, 1, 89.99),
(18, 1, 1, 1299.99),
(19, 4, 1, 45.99);
               """)

cursor.execute("""
               INSERT INTO customer_support_tickets (customer_id, ticket_date, issue_type, resolution_time_hours, satisfaction_rating) VALUES
(1, '2023-02-15', 'delivery', 24, 5),
(1, '2024-03-25', 'product_quality', 48, 3),
(2, '2023-06-25', 'delivery', 12, 5),
(3, '2024-02-20', 'refund', 72, 2),
(5, '2023-07-20', 'other', 6, 5),
(6, '2024-04-10', 'product_quality', 36, 4),
(7, '2023-03-05', 'delivery', 18, 5);
               
               """)

In [75]:

cursor.execute("""
with top_customer as (
SELECT c.customer_id
    FROM customers c
    join orders o on c.customer_id=o.customer_id
    WHERE c.registration_date < (DATE '2024-06-01' - INTERVAL '90 days')
    GROUP BY c.customer_id
    HAVING COUNT(o.order_id) >= 2),
  FEATURES AS(  
    SELECT 
    c.customer_id, 
    COUNT(o.order_id) AS total_orders,
    SUM(o.order_total) AS total_amount_spent,
    ROUND(AVG(o.order_total), 2) AS average_order_value,

    ROUND(
        DATE_PART('day', DATE '2024-06-01'::timestamp - MAX(o.order_date)::timestamp)
    ) AS days_since_last_order,
    SUM(case when o.status='cancelled' then 1 else 0 end)/count(o.order_id) as cancellation_rate,
    COUNT(DISTINCT p.category) AS unique_product_categories,
    COUNT(t.ticket_id) AS total_support_tickets,
    ROUND(AVG(t.satisfaction_rating), 2) AS avg_satisfaction_rating,

    -- active range
    ROUND(
        DATE_PART('day', MAX(o.order_date)::timestamp - MIN(o.order_date)::timestamp)
    ) AS days_active

FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN order_items oi ON o.order_id = oi.order_id
LEFT JOIN products p ON oi.product_id = p.product_id
LEFT JOIN customer_support_tickets t ON c.customer_id = t.customer_id

WHERE o.order_date <= DATE '2024-06-01' and c.customer_id IN (SELECT customer_id FROM top_customer)

GROUP BY c.customer_id),

 LABELS AS (
    select case when count(o.order_id)=0 then 1 else 0 end as churned,
    c.customer_id 
    from customers c
    left join orders o on c.customer_id=o.customer_id
    where c.customer_id IN (SELECT customer_id FROM top_customer) and o.order_date BETWEEN DATE '2024-06-01' AND DATE '2024-06-01' + INTERVAL '60 days'
    GROUP BY c.customer_id
)
    select f.*, COALESCE(l.churned,1) as churned
    from FEATURES f
    left join LABELS l on f.customer_id = l.customer_id;
    """)
customers_to_include= cursor.fetchall()
customers_to_include


[(1,
  10,
  Decimal('6211.86'),
  Decimal('621.19'),
  22.0,
  0,
  2,
  10,
  Decimal('4.00'),
  464.0,
  1),
 (2,
  3,
  Decimal('1435.97'),
  Decimal('478.66'),
  138.0,
  0,
  3,
  3,
  Decimal('5.00'),
  316.0,
  1),
 (3,
  3,
  Decimal('789.97'),
  Decimal('263.32'),
  108.0,
  0,
  2,
  3,
  Decimal('2.00'),
  310.0,
  1),
 (5,
  4,
  Decimal('2899.94'),
  Decimal('724.99'),
  245.0,
  0,
  2,
  4,
  Decimal('5.00'),
  152.0,
  0),
 (6,
  3,
  Decimal('739.97'),
  Decimal('246.66'),
  57.0,
  0,
  3,
  3,
  Decimal('4.00'),
  420.0,
  1),
 (7,
  2,
  Decimal('1345.98'),
  Decimal('672.99'),
  174.0,
  0,
  2,
  2,
  Decimal('5.00'),
  284.0,
  1)]